

run surface transforms in numrefields env

In [1]:
from nilearn import image
from nilearn import surface
import os.path as op
import numpy as np
import nibabel as nib

bids_folder_freesurfer = '/mnt_03/ds-dnumrisk'
bids_folder_data = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'

from numrisk.fmri_analysis.surface.utils import transform_surfaces

In [2]:

sub = '46' #'average'
confspec = '36Pscrub3BPfilter'
task = 'magjudge'
ses = 1
#bids_folder_data = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'

target_folder = op.join(bids_folder_data,'derivatives','networks_infomap_full_01')
fn_consens_mapping = op.join(target_folder, f'sub-{sub}_consensusMapping_confspec-{confspec}.npy')
consensus_labels = np.load(fn_consens_mapping)
np.shape(consensus_labels)

from numrisk.fmri_analysis.gradients.utils import get_basic_mask
mask, labeling_noParcel = get_basic_mask()

modules_fsav5 = np.full(mask.shape[0], np.nan, dtype=float)
modules_fsav5[mask] = consensus_labels
np.shape(modules_fsav5)

(20484,)

In [ ]:
source_space = 'fsaverage5'
target_space= 'fsaverage' # 'fsnative' #
#bids_folder_freesurfer = '/mnt_03/ds-dnumrisk'

target_folder = op.join(bids_folder_data,'derivatives','networks_infomap_full', f'sub-{sub}')
target_space=f'sub-{sub}' if target_space=='fsnative' else target_space

import os
os.makedirs(target_folder) if not op.exists(target_folder) else None

gm_both = np.split(modules_fsav5,2) # for i, hemi in enumerate(['L', 'R']): --> left first
for i, hemi in enumerate(['L', 'R']):
    gii_im_datar = nib.gifti.gifti.GiftiDataArray(data=gm_both[i].astype(np.float32))
    gii_im = nib.gifti.gifti.GiftiImage(darrays= [gii_im_datar])

    out_file = op.join(target_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{source_space}_hemi-{hemi}_consensus_labels.surf.gii')
    gii_im.to_filename(out_file) 

    fs_hemi = 'lh' if hemi=='L' else 'rh'
    transform_surfaces(in_file=out_file, fs_hemi=fs_hemi, bids_folder=bids_folder_freesurfer,
                            source_space= source_space, target_space=target_space)

251215-09:33:23,892 nipype.interface INFO:
	 stderr 2025-12-15T09:33:23.892325:** DA[0] has coordsys with intent NIFTI_INTENT_NONE (should be NIFTI_INTENT_POINTSET)
251215-09:33:24,843 nipype.interface INFO:
	 stdout 2025-12-15T09:33:24.843187:
251215-09:33:24,844 nipype.interface INFO:
	 stdout 2025-12-15T09:33:24.843187:7.3.2
251215-09:33:24,844 nipype.interface INFO:
	 stdout 2025-12-15T09:33:24.843187:
251215-09:33:24,844 nipype.interface INFO:
	 stdout 2025-12-15T09:33:24.843187:setenv SUBJECTS_DIR /mnt_03/ds-dnumrisk/derivatives/freesurfer
251215-09:33:24,845 nipype.interface INFO:
	 stdout 2025-12-15T09:33:24.843187:cd /home/ubuntu/git/parietal_patterns/networks_indTopology
251215-09:33:24,845 nipype.interface INFO:
	 stdout 2025-12-15T09:33:24.843187:mri_surf2surf --hemi lh --tval /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk/derivatives/networks_infomap_full/sub-46/sub-46_ses-1_task-magjudge_space-fsnative_hemi-L_consensus_labels.surf.gii --sval /mnt_AdaBD_largefi

In [ ]:
# Gradient maps
#sub = '33'

grad_n = 2 # -1 for indexing numpy array ;) ! 
ses = 1
task = 'magjudge'
source_space = 'fsaverage5' #'fsaverage5' misspelled before!
target_space= 'fsnative' #'fsaverage' # 
bids_folder_freesurfer = '/mnt_03/ds-dnumrisk'

target_folder = op.join(bids_folder_data,'derivatives','gradients.36Pscrub3BPfilterrunFD104', f'sub-{sub}')
target_space=f'sub-{sub}' if target_space=='fsnative' else target_space

gms = np.load(op.join(target_folder, f'sub-{sub}_g-aligned_space-fsaverag5_n10.npy')) # misspelled before!!
gm_both = np.split(gms[grad_n-1],2) # for i, hemi in enumerate(['L', 'R']): --> left first
for i, hemi in enumerate(['L', 'R']):
    gii_im_datar = nib.gifti.gifti.GiftiDataArray(data=gm_both[i].astype(np.float32))
    gii_im = nib.gifti.gifti.GiftiImage(darrays= [gii_im_datar])

    out_file = op.join(target_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{source_space}_hemi-{hemi}_grad-{grad_n}.surf.gii')
    gii_im.to_filename(out_file) 

    fs_hemi = 'lh' if hemi=='L' else 'rh'
    transform_surfaces(in_file=out_file, fs_hemi=fs_hemi, bids_folder=bids_folder_freesurfer,
                            source_space= source_space, target_space=target_space)